# 30 — Gold marts

Builds the curated marts consumed by Power BI and the EBA reporter:

| Mart | Grain | Consumer |
|---|---|---|
| `gold.fraud_kpis_daily` | day × country × instrument | exec dashboard |
| `gold.eba_report_q` | quarter × country × instrument × channel × exemption × fraud_type × geo × bearer | EBA reporter + dashboard |
| `gold.psd2_exemption_mix` | day × country × exemption | exec dashboard |
| `gold.scoring_latency_p99` | day × model_version | operational dashboard |

In [ ]:
from datetime import date

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()

process_date = spark.conf.get("pipeline.process_date", date.today().isoformat())
quarter = spark.conf.get("pipeline.quarter", "")  # e.g. '2025-Q1'
if not quarter:
    d = date.fromisoformat(process_date)
    quarter = f"{d.year}-Q{((d.month - 1) // 3) + 1}"
print(f"process_date={process_date} quarter={quarter}")

tx = spark.table("lh_fraud.silver.transactions")
dec = spark.table("lh_fraud.silver.decisions")
cases = spark.table("lh_fraud.silver.cases")

In [ ]:
# ---------------------------------------------------------------------------
# gold.fraud_kpis_daily
# ---------------------------------------------------------------------------

joined = (
    tx.alias("t")
    .join(dec.alias("d"), "tx_id", "left")
    .join(cases.alias("c"), "tx_id", "left")
)

kpis = (
    joined
    .filter(F.col("t.booking_date") == F.lit(process_date))
    .groupBy(
        F.col("t.booking_date").alias("kpi_date"),
        F.col("t.issuer_country").alias("issuer_country"),
        F.col("t.instrument_type").alias("instrument_type"),
    )
    .agg(
        F.count("t.tx_id").alias("tx_count"),
        F.sum("t.amount_eur").cast("decimal(18,2)").alias("tx_value_eur"),
        F.sum(F.when(F.col("d.decision") == "decline", 1).otherwise(0)).alias("decline_count"),
        F.sum(F.when(F.col("c.confirmed"), 1).otherwise(0)).alias("fraud_count"),
        F.sum(F.when(F.col("c.confirmed"), F.col("t.amount_eur")).otherwise(0)).cast("decimal(18,2)").alias("fraud_value_eur"),
        F.sum(F.coalesce(F.col("c.confirmed_loss_eur"), F.lit(0))).cast("decimal(18,2)").alias("loss_value_eur"),
    )
    .withColumn("decline_rate", F.col("decline_count") / F.col("tx_count"))
    .withColumn(
        "fraud_rate_bps",
        F.when(F.col("tx_value_eur") > 0, F.col("fraud_value_eur") / F.col("tx_value_eur") * 10000).otherwise(0)
    )
)

(
    kpis.write.format("delta").mode("overwrite")
    .option("replaceWhere", f"kpi_date = DATE'{process_date}'")
    .partitionBy("issuer_country")
    .saveAsTable("lh_fraud.gold.fraud_kpis_daily")
)
print("gold.fraud_kpis_daily written")

In [ ]:
# ---------------------------------------------------------------------------
# gold.eba_report_q  — the dimensional cube the EBA reporter reads
# ---------------------------------------------------------------------------

FRAUD_TYPE_MAP = {
    "social_engineering": "manipulation_of_payer",
    "account_takeover": "unauthorised_issuance",
    "card_not_present_fraud": "unauthorised_issuance",
    "merchant_collusion": "unauthorised_modification",
    "lost_or_stolen": "lost_stolen",
    "counterfeit": "counterfeit",
    "card_not_received": "card_not_received",
}

fraud_map_col = F.create_map(*[x for kv in FRAUD_TYPE_MAP.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])

# Quarter window
year, q = quarter.split("-Q")
year, q = int(year), int(q)
q_start = date(year, (q - 1) * 3 + 1, 1)
q_end = date(year + (1 if q == 4 else 0), 1 if q == 4 else q * 3 + 1, 1)

eba_src = (
    tx.alias("t")
    .filter((F.col("t.booking_date") >= F.lit(q_start)) & (F.col("t.booking_date") < F.lit(q_end)))
    .filter(F.col("t.issuer_country").isin("SE", "NO", "DK", "FI", "EE"))
    .join(dec.alias("d"), "tx_id", "left")
    .join(cases.alias("c"), "tx_id", "left")
)

eba = (
    eba_src
    .withColumn("reporting_country", F.col("t.issuer_country"))
    .withColumn("quarter", F.lit(quarter))
    .withColumn("instrument", F.col("t.instrument_type"))
    .withColumn("channel", F.col("t.channel"))
    .withColumn("sca_exemption", F.coalesce(F.col("d.sca_exemption_code"), F.lit("sca_applied")))
    .withColumn("fraud_type", F.element_at(fraud_map_col, F.col("c.fraud_type")))
    .withColumn("counterparty_geo", F.when(F.col("t.is_eea"), F.lit("eea")).otherwise(F.lit("non_eea")))
    .withColumn("loss_bearer", F.coalesce(F.col("c.loss_allocation"), F.lit(None).cast("string")))
    .withColumn("pisp_initiated", F.lit(False))
    .groupBy(
        "reporting_country", "quarter", "instrument", "channel", "sca_exemption",
        "fraud_type", "counterparty_geo", "loss_bearer", "pisp_initiated",
    )
    .agg(
        F.count("t.tx_id").alias("tx_count"),
        F.sum("t.amount_eur").cast("decimal(18,2)").alias("tx_value_eur"),
        F.sum(F.when(F.col("c.confirmed"), 1).otherwise(0)).alias("fraud_count"),
        F.sum(F.when(F.col("c.confirmed"), F.col("t.amount_eur")).otherwise(0)).cast("decimal(18,2)").alias("fraud_value_eur"),
        F.sum(F.coalesce(F.col("c.confirmed_loss_eur"), F.lit(0))).cast("decimal(18,2)").alias("loss_value_eur"),
    )
)

(
    eba.write.format("delta").mode("overwrite")
    .option("replaceWhere", f"quarter = '{quarter}'")
    .partitionBy("reporting_country", "quarter")
    .saveAsTable("lh_fraud.gold.eba_report_q")
)
print("gold.eba_report_q written")

In [ ]:
# ---------------------------------------------------------------------------
# gold.psd2_exemption_mix
# ---------------------------------------------------------------------------

mix = (
    tx.alias("t")
    .filter(F.col("t.booking_date") == F.lit(process_date))
    .join(dec.alias("d"), "tx_id", "left")
    .join(cases.alias("c"), "tx_id", "left")
    .groupBy(
        F.col("t.booking_date").alias("kpi_date"),
        F.col("t.issuer_country").alias("issuer_country"),
        F.coalesce(F.col("d.sca_exemption_code"), F.lit("sca_applied")).alias("sca_exemption_code"),
    )
    .agg(
        F.count("t.tx_id").alias("tx_count"),
        F.sum("t.amount_eur").cast("decimal(18,2)").alias("tx_value_eur"),
        F.sum(F.when(F.col("c.confirmed"), 1).otherwise(0)).alias("fraud_count"),
    )
    .withColumn(
        "fraud_rate_bps",
        F.when(F.col("tx_value_eur") > 0,
               F.col("fraud_count").cast("double") / F.col("tx_count") * 10000).otherwise(0)
    )
)

(
    mix.write.format("delta").mode("overwrite")
    .option("replaceWhere", f"kpi_date = DATE'{process_date}'")
    .partitionBy("issuer_country")
    .saveAsTable("lh_fraud.gold.psd2_exemption_mix")
)
print("gold.psd2_exemption_mix written")

In [ ]:
# ---------------------------------------------------------------------------
# gold.scoring_latency_p99
# ---------------------------------------------------------------------------

lat = (
    dec
    .filter(F.col("decision_date") == F.lit(process_date))
    .groupBy(F.col("decision_date").alias("kpi_date"), "model_version")
    .agg(
        F.expr("percentile_approx(latency_ms, 0.50)").alias("p50_ms"),
        F.expr("percentile_approx(latency_ms, 0.95)").alias("p95_ms"),
        F.expr("percentile_approx(latency_ms, 0.99)").alias("p99_ms"),
        F.count("*").alias("sample_count"),
    )
)

(
    lat.write.format("delta").mode("overwrite")
    .option("replaceWhere", f"kpi_date = DATE'{process_date}'")
    .saveAsTable("lh_fraud.gold.scoring_latency_p99")
)
print("gold.scoring_latency_p99 written")